# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [1]:
event_log_name = "small"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")

Event log name small
LTN Rows path small_ltn_rows.pkl


In [2]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


In [3]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.smallevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(small_leaky_row_classes)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table
[<class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-10'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-25'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-50'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-100'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-150'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-200'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-250'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-300'>, <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-350'>]


In [4]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}_new_evaluator'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)


Created directory: d:\LTNcoder\.out\plots\small_evaluations_both_2025-08-07-11-59-07_new_evaluator
Deleted all rows from Evaluation and Model tables.


# Training

In [5]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [6]:
ads = [
        dict(ad=SmallDAE, fit_kwargs=dict(epochs=20, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=20, batch_size=100)) for LEAKY_ROW_CLASS 
        in small_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=20, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in small_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.smallencoder.SmallDAE'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-25'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-50'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-100'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-150'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-200'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.smallencoder.SmallDAE-Leaky-250'>, 'fit_kwargs': {'epochs': 20, 'batch_size': 100}}, {'ad': <class 'april.anomaly

Fitting ADs:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1/20
41/41 [==============================] - 1s 10ms/step - loss: 0.2105 - accuracy: 9.7943e-04 - val_loss: 0.1057 - val_accuracy: 0.0000e+00
Epoch 2/20
41/41 [==============================] - 0s 7ms/step - loss: 0.0226 - accuracy: 0.0223 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 3/20
41/41 [==============================] - 0s 7ms/step - loss: 0.0045 - accuracy: 0.1298 - val_loss: 0.0042 - val_accuracy: 0.0000e+00
Epoch 4/20
41/41 [==============================] - 0s 6ms/step - loss: 0.0044 - accuracy: 0.2236 - val_loss: 0.0041 - val_accuracy: 0.0000e+00
Epoch 5/20
41/41 [==============================] - 0s 7ms/step - loss: 0.0043 - accuracy: 0.3039 - val_loss: 0.0041 - val_accuracy: 0.0000e+00
Epoch 6/20
41/41 [==============================] - 0s 7ms/step - loss: 0.0043 - accuracy: 0.3345 - val_loss: 0.0040 - val_accuracy: 0.0000e+00
Epoch 7/20
41/41 [==============================] - 0s 6ms/step - loss: 0.0042 - accuracy: 0.3408 - val_loss: 0.0040 - val_accuracy

In [7]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'likelihood+': <class 'april.anomalydetection.boehmer.LikelihoodPlusAnomalyDetector'>, 'naive': <class 'april.anomalydetection.bezerra.NaiveAnomalyDetector'>, 'naive+': <class 'april.anomalydetection.bezerra.NaivePlusAnomalyDetector'>, 'one-class-svm': <class 'april.anomalydetection.basic.OneClassSVM'>, 'perfect': <class 'april.anomalydetection.basic.PerfectAnomalyDetector'>, 'random': <class 'april.anomalydetection.basic.RandomAnomalyDetector'>, 'sam

# Evaluation

In [8]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [9]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [10]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['small-0.3-1_smalldae-leaky-100_20250807-115936.427923', 'small-0.3-1_smalldae-leaky-10_20250807-115916.296142', 'small-0.3-1_smalldae-leaky-150_20250807-115943.337418', 'small-0.3-1_smalldae-leaky-200_20250807-115950.297772', 'small-0.3-1_smalldae-leaky-250_20250807-115957.082688', 'small-0.3-1_smalldae-leaky-25_20250807-115922.761225', 'small-0.3-1_smalldae-leaky-300_20250807-120004.175635', 'small-0.3-1_smalldae-leaky-350_20250807-120011.588717', 'small-0.3-1_smalldae-leaky-50_20250807-115929.737031', 'small-0.3-1_smalldae_20250807-115907.914418', 'small-0.3-1_smallltnfrozen-100_20250807-120127.029667', 'small-0.3-1_smallltnfrozen-10_20250807-120018.388930', 'small-0.3-1_smallltnfrozen-150_20250807-120149.448019', 'small-0.3-1_smallltnfrozen-200_20250807-120222.554223', 'small-0.3-1_smallltnfrozen-250_20250807-120256.040875', 'small-0.3-1_smallltnfrozen-25_20250807-120041.673590', 'small-0.3-1_smallltnfrozen-300_20250807-120327.097340', 'small-0.3-1_smallltnfrozen

Evaluate:   0%|          | 0/19 [00:00<?, ?it/s]

Evaluating small-0.3-1_smalldae-leaky-100_20250807-115936.427923...
Loading model small-0.3-1_smalldae-leaky-100_20250807-115936.427923 / <april.fs.ModelFile object at 0x0000029AE9825190> for event log small-0.3-1 at path d:\LTNcoder\.out\models\small-0.3-1_smalldae-leaky-100_20250807-115936.427923.keras
Self.ad_: <april.anomalydetection.smallencoder.SmallDAE-Leaky-100 object at 0x0000029B28FFCB80>
<april.smallevaluator.Evaluator object at 0x0000029AE98253A0> loaded.
e.model_file: d:\LTNcoder\.out\models\small-0.3-1_smalldae-leaky-100_20250807-115936.427923.keras
e.model_name: small-0.3-1_smalldae-leaky-100_20250807-115936.427923
e.eventlog_name: small-0.3-1
Filtering dataset to 462 LTN rows.
Indices: [3, 7, 34, 43, 55, 73, 77, 83, 103, 112, 113, 114, 121, 129, 132, 151, 152, 160, 169, 171, 177, 180, 191, 207, 236, 238, 255, 261, 262, 269, 325, 326, 391, 409, 413, 452, 458, 464, 477, 490, 496, 504, 533, 534, 543, 548, 550, 556, 571, 572, 583, 623, 626, 636, 638, 641, 648, 649, 667, 684

In [11]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/5472 [00:00<?, ?it/s]

In [12]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['SmallDAE', 'SmallDAE-Leaky-10', 'SmallDAE-Leaky-25', 'SmallDAE-Leaky-50', 'SmallDAE-Leaky-100', 'SmallDAE-Leaky-150', 'SmallDAE-Leaky-200', 'SmallDAE-Leaky-250', 'SmallDAE-Leaky-300', 'SmallDAE-Leaky-350', 'SmallLTNFROZEN-10', 'SmallLTNFROZEN-25', 'SmallLTNFROZEN-50', 'SmallLTNFROZEN-100', 'SmallLTNFROZEN-150', 'SmallLTNFROZEN-200', 'SmallLTNFROZEN-250', 'SmallLTNFROZEN-300', 'SmallLTNFROZEN-350']


In [13]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [14]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [15]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [16]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [17]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

C:\Users\devas\AppData\Local\Temp\ipykernel_62556\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_62556\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


    axis                  ad process_model dataset_name        f1  precision  \
0   Case            SmallDAE         Small  small-0.3-1  0.486568   0.688889   
1   Case   SmallDAE-Leaky-10         Small  small-0.3-1  0.455861   0.673913   
2   Case  SmallDAE-Leaky-100         Small  small-0.3-1  0.456028   0.675000   
3   Case  SmallDAE-Leaky-150         Small  small-0.3-1  0.388782   0.686047   
4   Case  SmallDAE-Leaky-200         Small  small-0.3-1  0.423680   0.700000   
5   Case   SmallDAE-Leaky-25         Small  small-0.3-1  0.454609   0.723684   
6   Case  SmallDAE-Leaky-250         Small  small-0.3-1  0.408470   0.681818   
7   Case  SmallDAE-Leaky-300         Small  small-0.3-1  0.424727   0.688889   
8   Case  SmallDAE-Leaky-350         Small  small-0.3-1  0.415302   0.702381   
9   Case   SmallDAE-Leaky-50         Small  small-0.3-1  0.433409   0.657407   
10  Case   SmallLTNFROZEN-10         Small  small-0.3-1  0.586795   0.639098   
11  Case  SmallLTNFROZEN-100         Sma

In [18]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,SmallDAE,Small,small-0.3-1,0.486568,0.688889,0.376109
1,Case,SmallDAE-Leaky-10,Small,small-0.3-1,0.455861,0.673913,0.344420
2,Case,SmallDAE-Leaky-100,Small,small-0.3-1,0.456028,0.675000,0.344327
3,Case,SmallDAE-Leaky-150,Small,small-0.3-1,0.388782,0.686047,0.271249
4,Case,SmallDAE-Leaky-200,Small,small-0.3-1,0.423680,0.700000,0.303769
5,Case,SmallDAE-Leaky-25,Small,small-0.3-1,0.454609,0.723684,0.331393
6,Case,SmallDAE-Leaky-250,Small,small-0.3-1,0.408470,0.681818,0.291574
7,Case,SmallDAE-Leaky-300,Small,small-0.3-1,0.424727,0.688889,0.307003
8,Case,SmallDAE-Leaky-350,Small,small-0.3-1,0.415302,0.702381,0.294808
9,Case,SmallDAE-Leaky-50,Small,small-0.3-1,0.433409,0.657407,0.323263
